# 📱 DjezzyBot — Colab Runbook (T4 GPU)

Multilingual (Arabic / French / English / Darija) **voice + text RAG chatbot** for Djezzy.
Open-source only, runs entirely on a Colab **T4 (15 GB VRAM)**.

**Before running:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

Run the cells top to bottom:
1. Install system packages (Tesseract + Chromium)
2. Install Python deps
3. First scrape + build the FAISS index
4. Run the 14 acceptance tests (collects real latency)
5. Launch the Gradio app (`share=True`)

Stack: Qwen2.5-7B (4-bit NF4) · multilingual-e5-base · FAISS · Whisper-medium · Coqui XTTS-v2 · Playwright + Tesseract OCR.

## 1. System packages
Tesseract OCR (with Arabic/French/English language packs) and the Chromium dependencies Playwright needs.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y tesseract-ocr tesseract-ocr-ara tesseract-ocr-fra tesseract-ocr-eng
!tesseract --version | head -1

## 2. Python dependencies
Installs everything in `requirements.txt`, then downloads the Chromium browser for Playwright.

In [ ]:
# If the project isn't already on the Colab VM, upload it or clone it, e.g.:
# !git clone <your-repo-url> DjezzyBot_v2
%cd /content/DjezzyBot_v2
!pip install -q -r requirements.txt

In [ ]:
# Install the Chromium browser used by the BFS crawler
!playwright install chromium
!playwright install-deps chromium

In [ ]:
# Sanity check: GPU + key imports
import torch
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
import config
print('LLM   :', config.LLM_MODEL_ID)
print('Embed :', config.EMBED_MODEL_ID)
print('Domains:', config.DOMAINS)

## 3. First scrape + build the index
Discovers Djezzy URLs (sitemap + Playwright BFS crawl + path hints), OCRs offer images, and builds the FAISS index. This is the slow step (several minutes) — it caches results to `data/`, so later runs reuse them.

In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s %(message)s')

import scraper, indexer
pages = scraper.run_scrape()
print(f'Scraped {len(pages)} pages, {sum(p["has_ocr"] for p in pages)} with OCR.')
store = indexer.build_index(pages)
print('Indexed vectors:', store.index.ntotal)

## 4. Acceptance tests + latency
Runs the 14 scenarios end-to-end. Every call flows through the latency wrapper, so by the end the real timing numbers for `REPORT.md` are collected in `bot.LATENCY` / `latency_store.json`.

In [ ]:
import bot, test_scenarios
results = test_scenarios.run_all(store)
test_scenarios._summary(results)
print('Latency records:', len(bot.LATENCY))

## 5. Launch the app
Two-tab Gradio UI (text + voice). `boot()` loads the cached index and arms the daily 03:00 refresh; `share=True` gives a public link.

In [ ]:
import app
app.main()

## 6. (Optional) Manual checks
Quick one-off questions, and a manual refresh.

In [ ]:
import bot
for q in ['Quelles sont vos offres ?', "j'ai 500 DA", 'What plans do you have?', 'وش عندكم عروض']:
    r = bot.answer(q, store)
    print('\nQ:', q, '\nlang/route:', r['lang'], r['route'], '\nA:', r['text'][:300])

In [ ]:
# Force a refresh (same path as the daily 03:00 job and the UI button)
import scheduler
print(scheduler.force_refresh())